# 3.1 Fundamentals of Agentic AI

**Week 4 — Agentic AI & Multi-Agent Systems**

## Learning objectives
By the end of this notebook you will be able to:
- Define what makes an AI system "agentic" as opposed to a plain prompt/response system
- Explain the difference between traditional, task-focused AI and goal-driven autonomous AI
- Describe the **"Brain + Hands"** analogy for how LLM-based agents work
- Implement a minimal **ReAct** (Reason + Act) loop from scratch, without any framework

> This notebook uses small, dependency-free simulations so every cell runs offline. The same
> patterns map directly onto real LLM calls (OpenAI / Anthropic APIs) — those call-outs are shown
> in comments so you can swap in a real API key later.


## 1. What is Agentic AI?

A traditional LLM call is **stateless and single-shot**: you send a prompt, you get a completion,
the interaction ends. The model has no ability to:
- check whether its answer was actually correct,
- go look something up,
- take an action in the world,
- decide *by itself* what to do next.

**Agentic AI** wraps an LLM in a loop that lets it:
1. **Reason** about a goal and the current state,
2. **Decide** on an action (often: call a tool),
3. **Observe** the result of that action,
4. **Repeat** until the goal is satisfied or a stopping condition is hit.

| | Traditional AI | Agentic AI |
|---|---|---|
| Interaction | Single request → single response | Multi-step loop until goal is met |
| Control | Human decides every next step | Model decides its own next step |
| Scope | Answers a question | Pursues a *goal* |
| Failure mode | Wrong answer, conversation ends | Can retry, re-plan, or ask for help |
| Example | "Summarise this email" | "Get this invoice paid and confirm receipt" |


In [ ]:
# A traditional (non-agentic) call: one prompt in, one completion out. No loop, no tools.

def traditional_llm_call(prompt: str) -> str:
    """Pretend LLM call — in reality this would be:
    response = client.messages.create(model="claude-sonnet-4-6", max_tokens=200,
                                       messages=[{"role": "user", "content": prompt}])
    """
    canned_answers = {
        "what is the capital of france?": "The capital of France is Paris."
    }
    return canned_answers.get(prompt.strip().lower(), "I don't know how to answer that in one shot.")

print(traditional_llm_call("What is the capital of France?"))


Notice that the function above has **no way to check its own work, no way to take an action,
and no memory of anything beyond this one call.** That's the ceiling of non-agentic AI.

Now let's build the simplest possible agentic loop.


## 2. The "Brain + Hands" Analogy

Think of an agent as having two separate parts:

- **The Brain (the LLM):** does the reasoning. Given a goal and everything observed so far, it decides
  *what to do next* — in plain language or as a structured "tool call".
- **The Hands (the tools / environment):** actually *do* things — call an API, run a calculation,
  query a database, search the web, write a file. The brain cannot directly touch the world; it can only
  ask the hands to act, and read back what happened.

This separation is important: it's what lets you swap in different tools (hands) without retraining the
model (brain), and it's what makes agent behaviour inspectable — you can log every "thought" and every
"action" separately.


In [ ]:
from dataclasses import dataclass, field
from typing import Callable, Dict, Any, List

@dataclass
class Tool:
    name: str
    description: str
    func: Callable[..., Any]

class Brain:
    """Stands in for the LLM. Given a goal + history, decides the next reasoning step and action.
    In a real system, this method's body is replaced by an actual LLM call whose prompt includes
    the goal, the available tool descriptions, and the transcript so far."""

    def __init__(self, tools: Dict[str, Tool]):
        self.tools = tools

    def think(self, goal: str, history: List[str]) -> Dict[str, Any]:
        # --- Toy reasoning policy, standing in for a real LLM call ---
        if "weather" in goal.lower() and not any("weather_lookup" in h for h in history):
            return {"thought": "I need current weather data before I can answer.",
                     "action": "weather_lookup", "action_input": "Mumbai"}
        if "weather" in goal.lower():
            return {"thought": "I now have the weather data, I can answer the user.",
                     "action": "final_answer",
                     "action_input": "It's 31°C and humid in Mumbai right now."}
        return {"thought": "This goal needs no tools.", "action": "final_answer",
                 "action_input": "Done."}

print("Brain and Tool classes defined.")


## 3. A Minimal ReAct Loop (Reason + Act)

The **ReAct** pattern interleaves reasoning ("Thought: ...") with acting ("Action: ...") and
observing ("Observation: ...") in a loop, until the model produces a final answer. This is the
foundational control-flow pattern behind almost every agent framework (LangChain agents, AutoGen,
CrewAI all implement a variant of this loop under the hood).


In [ ]:
def weather_lookup(city: str) -> str:
    """A 'Hand' — a real tool would call a weather API here."""
    fake_weather_db = {"mumbai": "31°C, humid, 70% chance of rain"}
    return fake_weather_db.get(city.lower(), "No data for that city.")

tools = {"weather_lookup": Tool("weather_lookup", "Look up current weather for a city", weather_lookup)}
brain = Brain(tools)

def run_react_loop(goal: str, max_steps: int = 5):
    history = []
    print(f"GOAL: {goal}\n")
    for step in range(max_steps):
        decision = brain.think(goal, history)
        print(f"Step {step+1}")
        print(f"  Thought: {decision['thought']}")

        if decision["action"] == "final_answer":
            print(f"  Final Answer: {decision['action_input']}")
            return decision["action_input"]

        tool = tools[decision["action"]]
        observation = tool.func(decision["action_input"])
        print(f"  Action: {decision['action']}(\"{decision['action_input']}\")")
        print(f"  Observation: {observation}\n")
        history.append(f"{decision['action']} -> {observation}")

    print("Max steps reached without a final answer.")
    return None

run_react_loop("What's the weather like right now?")


### What just happened?

1. The **brain** looked at the goal and decided it lacked information → chose the `weather_lookup` tool.
2. The **hand** executed the tool and returned an observation.
3. The **brain** saw the new observation in its history and this time produced a `final_answer`.
4. The loop terminated because the model itself decided it was done — nobody told it "call the weather
   tool then stop." That self-directed control flow is the defining feature of agentic AI.

Try it yourself: change the goal so it doesn't mention "weather" and re-run. The brain should skip the
tool call entirely and answer directly — showing that reasoning, not a hardcoded script, drives the path.


In [ ]:
# Exercise: run with a goal that needs no tool call
run_react_loop("Just say hello to the user.")


## 4. Why This Matters for Real LLM Agents

In a production agent (e.g. built with the OpenAI or Anthropic API), the `Brain.think()` method above
is replaced by a real model call where:

- the **system prompt** describes the goal and the tools available (name, description, input schema),
- the **user/assistant transcript** carries the running history of thoughts/actions/observations,
- the model's response is parsed for either a **tool call** (structured function-calling output) or a
  **final answer**.

```python
# Sketch of a real single ReAct step using the Anthropic API's tool-use feature
response = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=500,
    tools=[{"name": "weather_lookup", "description": "Get current weather for a city",
            "input_schema": {"type": "object", "properties": {"city": {"type": "string"}}}}],
    messages=transcript,
)
# response.content will contain either a tool_use block or a text block (final answer)
```

## 5. Key Takeaways

- Agentic AI = an LLM **inside a loop** that reasons, acts, observes, and decides when it's done.
- The **Brain + Hands** split separates *deciding* (LLM) from *doing* (tools/environment).
- **ReAct** (Reason → Act → Observe → repeat) is the foundational control pattern behind every agent
  framework you'll meet later this week (LangChain, AutoGen, CrewAI).
- The next notebook (3.2) builds a first *real* agent using LangChain's tool abstraction and introduces
  AutoGen's two core agent roles.

## Check your understanding
1. What is the key difference between a traditional LLM call and an agentic loop?
2. In the Brain + Hands analogy, which part decides *what* to do, and which part *does* it?
3. Why does the ReAct loop need an explicit "final_answer" action instead of just running forever?
